# Proyecto Laboratorio de Modelación II

## Data Sintética

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

import json
from pathlib import Path

Aquí hay tres familias de instancias:

* small
* medium
* large

Cada una tiene parámetros distintos:

* seed: semilla para reproducibilidad
* n_I: cursos del bloque 1
* n_K: cursos del bloque 2
* C: tamaño común por curso
* split_level: cuán dispersos son los flujos
* generation_mode: cómo generar los flujos
* free_rate: proporción de libres, si se usa modo flexible
* force_nR_equals_nI: para imponer que número de salas = número de cursos del bloque 1

Generación de Instancias -> Cantidad pequeña, mediana y grande de datos

In [3]:
INSTANCE_CONFIGS = {
    "small": {
        "seed": 42,
        "n_I": 8,
        "n_K": 6,
        "C": 30,
        "split_level": "low",          # low, medium, high
        "generation_mode": "column_balanced",  # column_balanced o row_balanced
        "free_rate": 0.15,             # solo se usa en row_balanced
        "force_nR_equals_nI": True
    },
    "medium": {
        "seed": 42,
        "n_I": 15,
        "n_K": 12,
        "C": 40,
        "split_level": "medium",
        "generation_mode": "column_balanced",
        "free_rate": 0.18,
        "force_nR_equals_nI": True
    },
    "large": {
        "seed": 42,
        "n_I": 25,
        "n_K": 20,
        "C": 50,
        "split_level": "high",
        "generation_mode": "column_balanced",
        "free_rate": 0.20,
        "force_nR_equals_nI": True
    }
}

Split level: Cómo se interpreta
* low

  Cada fila de la matriz de movilidad tiende a tener pocos destinos no nulos. O sea, un curso del bloque 1 manda estudiantes a 1 o 2 cursos principalmente.

* medium

  La cohorte se reparte entre una cantidad intermedia de destinos.

* high

  Los estudiantes se dispersan más; una fila puede tener más columnas no nulas.

* A. row_balanced

  Obliga a que cada fila respete el total C.

  Es decir, para cada curso del bloque 1:
  $$ \sum_{k} f_{ik} + f_{iL} =C $$

  Entonces puede pasar que:

  un curso del bloque 2 reciba 18 estudiantes
  otro reciba 44
  otro reciba 27

* B. column_balanced
  Además de respetar cada fila, obliga a que cada columna sume exactamente C.

  O sea:
  $$ \sum_k f_ik =C$$
  para cada curso k del bloque 2.

  Consecuencia importante

  Si n_I > n_K, entonces sobran estudiantes del bloque 1 y esos deben ir a free_vector, entonces se tiene que los estudiantes libres son multiplos de C




* free rate: Define la proporción esperada de estudiantes que quedarán libres solo en el modo row_balanced. En modo column balanced, los libres se generan por consistencia global, por lo tanto no se usa.

Funciones Auxiliares

get_split_params():El parámetro alpha controla qué tan equilibrado o concentrado será el reparto cuando usamos una distribución Dirichlet (Siempre devuelve proporciones que suman exactamente 1)

* alpha pequeño → reparto más concentrado, eso sirve para instancias donde cada curso del bloque 1 alimenta casi siempre a uno o dos destinos principales.

* alpha grande → reparto más uniforme, eso sirve para que las cohortes se dispersen más.



In [4]:
def resolve_config(config: dict) -> dict:
    cfg = dict(config)
    if cfg.get("force_nR_equals_nI", True):
        cfg["n_R"] = cfg["n_I"]
    else:
        cfg["n_R"] = int(cfg["n_R"])
    return cfg


def get_rng(seed: int) -> np.random.Generator:
    return np.random.default_rng(seed)


def get_split_params(split_level: str, n_k: int) -> dict:
    if n_k <= 0:
        raise ValueError("n_K debe ser mayor que 0.")

    split_level = split_level.lower().strip()

    if split_level == "low":
        return {
            "dest_min": 1,
            "dest_max": min(2, n_k),
            "alpha": 0.5
        }
    elif split_level == "medium":
        return {
            "dest_min": min(2, n_k),
            "dest_max": min(4, n_k),
            "alpha": 1.0
        }
    elif split_level == "high":
        return {
            "dest_min": min(3, n_k),
            "dest_max": min(6, n_k),
            "alpha": 2.0
        }
    else:
        raise ValueError("split_level debe ser 'low', 'medium' o 'high'.")


def choose_destination_count(n_k: int, split_level: str, rng: np.random.Generator) -> int:
    params = get_split_params(split_level, n_k)
    if params["dest_min"] == params["dest_max"]:
        return params["dest_min"]
    return int(rng.integers(params["dest_min"], params["dest_max"] + 1))


def build_cost_matrix(n_r: int) -> np.ndarray:
    """
    Matriz binaria:
    c_rs = 0 si r = s
    c_rs = 1 si r != s
    """
    cost = np.ones((n_r, n_r), dtype=int)
    np.fill_diagonal(cost, 0)
    return cost


def distribute_total_with_cap(total: int, n_bins: int, cap: int, rng: np.random.Generator) -> np.ndarray:
    """
    Distribuye 'total' unidades enteras en 'n_bins' posiciones,
    con máximo 'cap' por posición.
    """
    if total < 0:
        raise ValueError("total no puede ser negativo.")
    if n_bins <= 0:
        raise ValueError("n_bins debe ser positivo.")
    if cap < 0:
        raise ValueError("cap no puede ser negativo.")
    if total > n_bins * cap:
        raise ValueError("No se puede distribuir total con el cap dado.")

    values = np.zeros(n_bins, dtype=int)
    remaining = total

    while remaining > 0:
        valid = np.where(values < cap)[0]
        idx = int(rng.choice(valid))
        max_add = min(cap - values[idx], remaining)
        add = int(rng.integers(1, max_add + 1))
        values[idx] += add
        remaining -= add

    return values


def allocate_unconstrained(total: int, n_k: int, chosen_cols: np.ndarray, alpha: float, rng: np.random.Generator) -> np.ndarray:
    """
    Distribuye 'total' entre un subconjunto de columnas, sin restricción de capacidad por columna.
    """
    row = np.zeros(n_k, dtype=int)

    if total == 0:
        return row

    weights = rng.dirichlet(np.full(len(chosen_cols), alpha))
    alloc = rng.multinomial(total, weights)
    row[chosen_cols] = alloc
    return row


def allocate_with_caps(total: int, remaining_caps: np.ndarray, chosen_cols: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    """
    Distribuye 'total' entre chosen_cols respetando las capacidades restantes por columna.
    """
    row = np.zeros(len(remaining_caps), dtype=int)

    if total == 0:
        return row

    caps = remaining_caps[chosen_cols].copy()
    if caps.sum() < total:
        raise ValueError("Las columnas elegidas no tienen capacidad suficiente.")

    local_alloc = np.zeros(len(chosen_cols), dtype=int)
    remaining = total

    while remaining > 0:
        local_caps_remaining = caps - local_alloc
        valid = np.where(local_caps_remaining > 0)[0]
        probs = local_caps_remaining[valid] / local_caps_remaining[valid].sum()
        j = int(rng.choice(valid, p=probs))
        local_alloc[j] += 1
        remaining -= 1

    row[chosen_cols] = local_alloc
    return row

Generación del vector libre

In [5]:
def generate_free_vector(cfg: dict, rng: np.random.Generator) -> np.ndarray:
    n_i = cfg["n_I"]
    n_k = cfg["n_K"]
    C = cfg["C"]
    mode = cfg["generation_mode"]
    free_rate = cfg.get("free_rate", 0.0)

    if mode == "column_balanced":
        # Para forzar que todas las columnas sumen C:
        # total_flow = n_K * C
        # total_students_block1 = n_I * C
        # total_free = C * (n_I - n_K)
        if n_i < n_k:
            raise ValueError(
                "En mode='column_balanced' se requiere n_I >= n_K "
                "para que existan libres y todas las columnas puedan sumar C."
            )

        total_free = C * (n_i - n_k)
        free_vector = distribute_total_with_cap(total_free, n_i, C, rng)
        return free_vector.astype(int)

    elif mode == "row_balanced":
        free_vector = rng.binomial(n=C, p=free_rate, size=n_i)
        return free_vector.astype(int)

    else:
        raise ValueError("generation_mode debe ser 'column_balanced' o 'row_balanced'.")

Generación de Matriz de Flujos

In [6]:
def generate_flow_matrix_row_balanced(cfg: dict, free_vector: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    n_i = cfg["n_I"]
    n_k = cfg["n_K"]
    C = cfg["C"]
    split_level = cfg["split_level"]

    flow = np.zeros((n_i, n_k), dtype=int)
    params = get_split_params(split_level, n_k)

    all_cols = np.arange(n_k)

    for i in range(n_i):
        row_total = C - int(free_vector[i])

        if row_total == 0:
            continue

        dest_count = choose_destination_count(n_k, split_level, rng)
        chosen_cols = rng.choice(all_cols, size=dest_count, replace=False)

        row_alloc = allocate_unconstrained(
            total=row_total,
            n_k=n_k,
            chosen_cols=chosen_cols,
            alpha=params["alpha"],
            rng=rng
        )
        flow[i, :] = row_alloc

    return flow


def generate_flow_matrix_column_balanced(cfg: dict, free_vector: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    n_i = cfg["n_I"]
    n_k = cfg["n_K"]
    C = cfg["C"]
    split_level = cfg["split_level"]

    flow = np.zeros((n_i, n_k), dtype=int)
    remaining_col = np.full(n_k, C, dtype=int)
    row_totals = C - free_vector

    row_order = rng.permutation(n_i)

    for pos, i in enumerate(row_order):
        row_total = int(row_totals[i])

        if pos == len(row_order) - 1:
            # La última fila absorbe exactamente lo que quede
            if remaining_col.sum() != row_total:
                raise ValueError(
                    f"Inconsistencia final: remaining_col.sum()={remaining_col.sum()} "
                    f"pero row_total={row_total}."
                )
            flow[i, :] = remaining_col
            remaining_col = remaining_col - flow[i, :]
            continue

        if row_total == 0:
            continue

        positive_cols = np.where(remaining_col > 0)[0]
        dest_count = choose_destination_count(len(positive_cols), split_level, rng)
        chosen_cols = rng.choice(positive_cols, size=dest_count, replace=False)

        # Si el subconjunto no alcanza, lo expandimos con columnas de mayor capacidad restante
        if remaining_col[chosen_cols].sum() < row_total:
            missing_cols = [c for c in positive_cols if c not in set(chosen_cols)]
            missing_cols = sorted(missing_cols, key=lambda c: remaining_col[c], reverse=True)

            chosen_list = list(chosen_cols)
            for c in missing_cols:
                chosen_list.append(c)
                if remaining_col[chosen_list].sum() >= row_total:
                    break
            chosen_cols = np.array(chosen_list, dtype=int)

        row_alloc = allocate_with_caps(
            total=row_total,
            remaining_caps=remaining_col,
            chosen_cols=chosen_cols,
            rng=rng
        )

        flow[i, :] = row_alloc
        remaining_col -= row_alloc

    if remaining_col.sum() != 0:
        raise ValueError("La generación column_balanced terminó con capacidad remanente no asignada.")

    return flow


def generate_flow_matrix(cfg: dict, free_vector: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    mode = cfg["generation_mode"]

    if mode == "column_balanced":
        return generate_flow_matrix_column_balanced(cfg, free_vector, rng)
    elif mode == "row_balanced":
        return generate_flow_matrix_row_balanced(cfg, free_vector, rng)
    else:
        raise ValueError("generation_mode no reconocido.")

Construccion Completa de la instancia

In [7]:
def build_instance(config: dict) -> dict:
    cfg = resolve_config(config)
    rng = get_rng(cfg["seed"])

    cost_matrix = build_cost_matrix(cfg["n_R"])
    free_vector = generate_free_vector(cfg, rng)
    flow_matrix = generate_flow_matrix(cfg, free_vector, rng)

    instance = {
        "metadata": {
            "seed": cfg["seed"],
            "n_I": cfg["n_I"],
            "n_K": cfg["n_K"],
            "n_R": cfg["n_R"],
            "C": cfg["C"],
            "split_level": cfg["split_level"],
            "generation_mode": cfg["generation_mode"],
            "free_rate": cfg.get("free_rate", None),
            "force_nR_equals_nI": cfg.get("force_nR_equals_nI", True)
        },
        "cost_matrix": cost_matrix,
        "flow_matrix": flow_matrix,
        "free_vector": free_vector
    }
    return instance

Funcion de Resumen de la instancia

In [8]:
def summarize_instance(instance: dict) -> pd.DataFrame:
    meta = instance["metadata"]
    flow = instance["flow_matrix"]
    free = instance["free_vector"]

    row_sums = flow.sum(axis=1)
    col_sums = flow.sum(axis=0)
    nonzero_per_row = (flow > 0).sum(axis=1)

    summary = {
        "n_I": meta["n_I"],
        "n_K": meta["n_K"],
        "n_R": meta["n_R"],
        "C": meta["C"],
        "generation_mode": meta["generation_mode"],
        "split_level": meta["split_level"],
        "total_students_block1": meta["n_I"] * meta["C"],
        "total_students_to_block2": int(flow.sum()),
        "total_free_students": int(free.sum()),
        "avg_free_per_course_i": float(free.mean()),
        "avg_nonzero_destinations_per_row": float(nonzero_per_row.mean()),
        "min_column_sum": int(col_sums.min()),
        "max_column_sum": int(col_sums.max())
    }

    return pd.DataFrame([summary])

Ejecutar Generación

In [10]:
INSTANCE_NAME = "small"  # "small", "medium" o "large"

config = INSTANCE_CONFIGS[INSTANCE_NAME]
instance = build_instance(config)
display(summarize_instance(instance))

,n_I,n_K,n_R,C,generation_mode,split_level,total_students_block1,total_students_to_block2,total_free_students,avg_free_per_course_i,avg_nonzero_destinations_per_row,min_column_sum,max_column_sum
0,8,6,8,30,column_balanced,low,240,180,60,7.5,2.0,30,30


Vista Rapida de Matrices

In [11]:
meta = instance["metadata"]

print("Metadata")
print(json.dumps(meta, indent=4))

print("\nCost matrix")
display(pd.DataFrame(
    instance["cost_matrix"],
    index=[f"R{r+1}" for r in range(meta["n_R"])],
    columns=[f"R{r+1}" for r in range(meta["n_R"])]
))

print("\nFlow matrix")
display(pd.DataFrame(
    instance["flow_matrix"],
    index=[f"I{i+1}" for i in range(meta["n_I"])],
    columns=[f"K{k+1}" for k in range(meta["n_K"])]
))

print("\nFree vector")
display(pd.DataFrame({
    "curso_i": [f"I{i+1}" for i in range(meta["n_I"])],
    "free_students": instance["free_vector"]
}))

Metadata
{
    "seed": 42,
    "n_I": 8,
    "n_K": 6,
    "n_R": 8,
    "C": 30,
    "split_level": "low",
    "generation_mode": "column_balanced",
    "free_rate": 0.15,
    "force_nR_equals_nI": true
}

Cost matrix


,R1,R2,R3,R4,R5,R6,R7,R8
R1,0,1,1,1,1,1,1,1
R2,1,0,1,1,1,1,1,1
R3,1,1,0,1,1,1,1,1
R4,1,1,1,0,1,1,1,1
R5,1,1,1,1,0,1,1,1
R6,1,1,1,1,1,0,1,1
R7,1,1,1,1,1,1,0,1
R8,1,1,1,1,1,1,1,0



Flow matrix


,K1,K2,K3,K4,K5,K6
I1,1,0,0,0,2,0
I2,6,24,0,0,0,0
I3,14,0,0,0,0,16
I4,0,0,8,0,0,3
I5,1,0,0,0,18,11
I6,0,6,0,0,10,0
I7,8,0,22,0,0,0
I8,0,0,0,30,0,0



Free vector


,curso_i,free_students
0,I1,27
1,I2,0
2,I3,0
3,I4,19
4,I5,0
5,I6,14
6,I7,0
7,I8,0


In [12]:
def prepare_instance_for_validation(instance: dict) -> dict:
    metadata = instance["metadata"]
    cost_matrix = instance["cost_matrix"]
    flow_matrix = instance["flow_matrix"]
    free_vector = instance["free_vector"]

    n_i = metadata["n_I"]
    n_k = metadata["n_K"]
    n_r = metadata["n_R"]

    cost_df = pd.DataFrame(
        cost_matrix,
        index=[f"R{r+1}" for r in range(n_r)],
        columns=[f"R{r+1}" for r in range(n_r)]
    )

    flow_df = pd.DataFrame(
        flow_matrix,
        index=[f"I{i+1}" for i in range(n_i)],
        columns=[f"K{k+1}" for k in range(n_k)]
    )

    free_df = pd.DataFrame({
        "curso_i": [f"I{i+1}" for i in range(n_i)],
        "free_students": free_vector
    })

    return {
        "metadata": metadata,
        "cost_df": cost_df,
        "flow_df": flow_df,
        "free_df": free_df,
        "cost_matrix": cost_matrix,
        "flow_matrix": flow_matrix,
        "free_vector": free_vector
    }

In [13]:
def is_integer_array(arr: np.ndarray) -> bool:
    return np.all(np.equal(arr, np.floor(arr)))


def validate_shapes(data: dict) -> list[str]:
    errors = []
    meta = data["metadata"]
    cost = data["cost_matrix"]
    flow = data["flow_matrix"]
    free = data["free_vector"]

    n_i = meta["n_I"]
    n_k = meta["n_K"]
    n_r = meta["n_R"]

    if cost.shape != (n_r, n_r):
        errors.append(f"cost_matrix tiene shape {cost.shape}, pero se esperaba {(n_r, n_r)}.")

    if flow.shape != (n_i, n_k):
        errors.append(f"flow_matrix tiene shape {flow.shape}, pero se esperaba {(n_i, n_k)}.")

    if free.shape[0] != n_i:
        errors.append(f"free_vector tiene largo {free.shape[0]}, pero se esperaba {n_i}.")

    return errors


def validate_cost_matrix(data: dict) -> list[str]:
    errors = []
    cost = data["cost_matrix"]

    if cost.shape[0] != cost.shape[1]:
        errors.append("La matriz de costos no es cuadrada.")
        return errors

    if not np.array_equal(cost, cost.T):
        errors.append("La matriz de costos no es simétrica.")

    diag = np.diag(cost)
    if not np.all(diag == 0):
        errors.append("La diagonal de la matriz de costos no es completamente cero.")

    off_diag = cost[~np.eye(cost.shape[0], dtype=bool)]
    if not np.all(off_diag == 1):
        errors.append("Fuera de la diagonal, la matriz de costos no es completamente unitaria.")

    unique_vals = np.unique(cost)
    if not set(unique_vals).issubset({0, 1}):
        errors.append(f"La matriz de costos contiene valores fuera de {{0,1}}: {unique_vals}.")

    if np.any(cost < 0):
        errors.append("La matriz de costos contiene valores negativos.")

    if not is_integer_array(cost):
        errors.append("La matriz de costos contiene valores no enteros.")

    return errors


def validate_flow_and_free_integrality(data: dict) -> list[str]:
    errors = []
    flow = data["flow_matrix"]
    free = data["free_vector"]

    if np.any(flow < 0):
        errors.append("flow_matrix contiene valores negativos.")

    if np.any(free < 0):
        errors.append("free_vector contiene valores negativos.")

    if not is_integer_array(flow):
        errors.append("flow_matrix contiene valores no enteros.")

    if not is_integer_array(free):
        errors.append("free_vector contiene valores no enteros.")

    return errors


def validate_row_conservation(data: dict) -> list[str]:
    errors = []
    meta = data["metadata"]
    flow = data["flow_matrix"]
    free = data["free_vector"]
    C = meta["C"]

    row_totals = flow.sum(axis=1) + free

    bad_rows = np.where(row_totals != C)[0]
    if len(bad_rows) > 0:
        details = []
        for i in bad_rows:
            details.append(f"I{i+1}: suma fila + libres = {row_totals[i]} (esperado {C})")
        errors.append("No se cumple conservación por fila:\n" + "\n".join(details))

    return errors


def validate_column_totals_if_required(data: dict) -> list[str]:
    errors = []
    meta = data["metadata"]
    flow = data["flow_matrix"]
    mode = meta["generation_mode"]
    C = meta["C"]

    if mode == "column_balanced":
        col_totals = flow.sum(axis=0)
        bad_cols = np.where(col_totals != C)[0]
        if len(bad_cols) > 0:
            details = []
            for k in bad_cols:
                details.append(f"K{k+1}: suma columna = {col_totals[k]} (esperado {C})")
            errors.append("No se cumple balance por columna:\n" + "\n".join(details))

    return errors


def validate_rooms_equals_courses_if_required(data: dict) -> list[str]:
    errors = []
    meta = data["metadata"]

    if meta.get("force_nR_equals_nI", True):
        if meta["n_R"] != meta["n_I"]:
            errors.append(
                f"Se esperaba n_R = n_I, pero n_R={meta['n_R']} y n_I={meta['n_I']}."
            )

    return errors

In [14]:
def build_summary(data: dict) -> pd.DataFrame:
    meta = data["metadata"]
    flow = data["flow_matrix"]
    free = data["free_vector"]
    cost = data["cost_matrix"]

    row_sums = flow.sum(axis=1)
    col_sums = flow.sum(axis=0)
    nonzero_per_row = (flow > 0).sum(axis=1)

    summary = {
        "n_I": meta["n_I"],
        "n_K": meta["n_K"],
        "n_R": meta["n_R"],
        "C": meta["C"],
        "generation_mode": meta["generation_mode"],
        "split_level": meta["split_level"],
        "total_students_block1": meta["n_I"] * meta["C"],
        "students_to_block2": int(flow.sum()),
        "students_free": int(free.sum()),
        "free_rate_observed": float(free.sum() / (meta["n_I"] * meta["C"])),
        "avg_free_per_row": float(free.mean()),
        "min_row_flow_sum": int(row_sums.min()),
        "max_row_flow_sum": int(row_sums.max()),
        "min_col_flow_sum": int(col_sums.min()),
        "max_col_flow_sum": int(col_sums.max()),
        "avg_nonzero_destinations_per_row": float(nonzero_per_row.mean()),
        "cost_diag_all_zero": bool(np.all(np.diag(cost) == 0)),
        "cost_offdiag_all_one": bool(np.all(cost[~np.eye(cost.shape[0], dtype=bool)] == 1))
    }

    return pd.DataFrame([summary])

In [15]:
def run_all_validations(instance: dict) -> dict:
    data = prepare_instance_for_validation(instance)

    checks = {
        "shapes": validate_shapes(data),
        "cost_matrix": validate_cost_matrix(data),
        "flow_and_free_integrality": validate_flow_and_free_integrality(data),
        "row_conservation": validate_row_conservation(data),
        "column_totals_if_required": validate_column_totals_if_required(data),
        "rooms_equals_courses_if_required": validate_rooms_equals_courses_if_required(data)
    }

    errors = []
    for _, errs in checks.items():
        errors.extend(errs)

    result = {
        "ok": len(errors) == 0,
        "errors": errors,
        "summary": build_summary(data),
        "data": data
    }

    return result

In [16]:
def print_validation_report(report: dict) -> None:
    print("=" * 70)
    print("REPORTE DE VALIDACIÓN")
    print("=" * 70)

    if report["ok"]:
        print("Estado general: OK")
    else:
        print("Estado general: CON ERRORES")

    print("\nResumen de la instancia:")
    display(report["summary"])

    if report["errors"]:
        print("\nErrores encontrados:")
        for i, err in enumerate(report["errors"], start=1):
            print(f"{i}. {err}")
    else:
        print("\nNo se detectaron errores.")

In [17]:
report = run_all_validations(instance)
print_validation_report(report)

REPORTE DE VALIDACIÓN
Estado general: OK

Resumen de la instancia:


,n_I,n_K,n_R,C,generation_mode,split_level,total_students_block1,students_to_block2,students_free,free_rate_observed,avg_free_per_row,min_row_flow_sum,max_row_flow_sum,min_col_flow_sum,max_col_flow_sum,avg_nonzero_destinations_per_row,cost_diag_all_zero,cost_offdiag_all_one
0,8,6,8,30,column_balanced,low,240,180,60,0.25,7.5,3,30,30,30,2.0,True,True



No se detectaron errores.


In [18]:
data = report["data"]

print("Metadata")
print(json.dumps(data["metadata"], indent=4))

print("\nCost matrix")
display(data["cost_df"])

print("\nFlow matrix")
display(data["flow_df"])

print("\nFree vector")
display(data["free_df"])

Metadata
{
    "seed": 42,
    "n_I": 8,
    "n_K": 6,
    "n_R": 8,
    "C": 30,
    "split_level": "low",
    "generation_mode": "column_balanced",
    "free_rate": 0.15,
    "force_nR_equals_nI": true
}

Cost matrix


,R1,R2,R3,R4,R5,R6,R7,R8
R1,0,1,1,1,1,1,1,1
R2,1,0,1,1,1,1,1,1
R3,1,1,0,1,1,1,1,1
R4,1,1,1,0,1,1,1,1
R5,1,1,1,1,0,1,1,1
R6,1,1,1,1,1,0,1,1
R7,1,1,1,1,1,1,0,1
R8,1,1,1,1,1,1,1,0



Flow matrix


,K1,K2,K3,K4,K5,K6
I1,1,0,0,0,2,0
I2,6,24,0,0,0,0
I3,14,0,0,0,0,16
I4,0,0,8,0,0,3
I5,1,0,0,0,18,11
I6,0,6,0,0,10,0
I7,8,0,22,0,0,0
I8,0,0,0,30,0,0



Free vector


,curso_i,free_students
0,I1,27
1,I2,0
2,I3,0
3,I4,19
4,I5,0
5,I6,14
6,I7,0
7,I8,0


In [20]:
# Si tu notebook está dentro de la carpeta "notebooks/"
output_dir = Path("../data/modelo1")
output_dir.mkdir(parents=True, exist_ok=True)

# Matriz de costos
data["cost_df"].to_csv(output_dir / "cost_matrix.csv")

# Matriz de flujo
data["flow_df"].to_csv(output_dir / "flow_matrix.csv")

# Vector libre
data["free_df"].to_csv(output_dir / "free_vector.csv", index=False)  

print("Archivos guardados en:", output_dir.resolve())

Archivos guardados en: C:\Users\yuvi8\Desktop\Semestre Actual\Laboratorio de Modelacion II\data\modelo1
